# Inspect TOFs — single-shot electron / ion hits, binned

Single-shot TOF hits from a combined file (`write_h5.py`) → optional filtering → average TOF spectra in
bins of any per-shot quantity (GMD, hit counts, OPIS, time), with optional normalisation. With
`NORMALISE = "gmd"` (counts per µJ, `sum(counts)/sum(GMD)` per bin) a purely linear process gives the
same spectrum in every GMD bin; the ratio map in section 4 shows where it doesn't.

Per-shot data are plain arrays: `H` (hits of each shot, zero-padded, `(n_shots, max_hits)`) and the dict
`shots` of per-shot values — edit and add to them freely.

In [ ]:
# ---- Parameters ---------------------------------------------------------------
COMBINED_H5 = None           # combined file from write_h5.py (relative to config.COMBINED_DIR or absolute)
TRAIN_STEP = 1               # every N-th train
BUNCH_RANGE = None           # (start, stop) bunch slots used; None = slots where the GMD shows pulses
DETECTOR = "e"               # "e" = electrons (tofs_e), "i" = ions (tofs_i)
TOF_RANGE = (0, 4e4)         # TDC units (one bunch spans ~39877)
TOF_BIN = 20                 # TDC units per bin

# Filters on per-shot values: (lo, hi) or None. Keys of the `shots` dict (section 1).
FILTERS = {"gmd": None, "n_e": None, "n_i": None, "mpe": None, "t_min": None}

# Binning
BIN_ON = "gmd"               # key of `shots`: "gmd", "n_e", "n_i", "mpe", "t_min", "slot", ...
N_BINS = 8
BIN_PERCENTILES = (2, 98)    # edges spread evenly between these percentiles of BIN_ON ...
BIN_EDGES = None             # ... unless explicit edges are given
NORMALISE = "gmd"            # None: summed counts | "shot": counts per shot | "gmd": counts per µJ | "area": unit area
LOG_Y = True
RATIO_REBIN = 10             # section 4 map: merge this many TOF bins ...
RATIO_MIN_COUNTS = 50        # ... and blank cells with fewer hits than this

In [ ]:
# ---- Imports and paths ------------------------------------------------------
from pathlib import Path
import sys
import warnings

import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# Repo root = the first folder above the working directory that holds analysis/scripts/config.py
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "analysis" / "scripts" / "config.py").exists())
sys.path.insert(0, str(REPO_ROOT / "analysis" / "scripts"))
import binning
import config
import data_loading
import processing

# Empty bunch slots are NaN by design, so these numpy warnings are expected noise
for msg in ("All-NaN slice", "Mean of empty slice", "invalid value encountered", "divide by zero",
            "Degrees of freedom"):
    warnings.filterwarnings("ignore", message=msg)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
TRAIN_RATE_HZ = 10

## 1. Load

In [ ]:
assert COMBINED_H5 is not None, "set COMBINED_H5 — TOF hits only exist in combined files from write_h5.py"
h5_path = Path(COMBINED_H5)
if not h5_path.is_absolute():
    h5_path = config.COMBINED_DIR / h5_path
data = data_loading.load_data(str(h5_path), config=1, downsample_N=TRAIN_STEP, load_vls=False)
source = f"{h5_path.name} (every {TRAIN_STEP}. train)"
assert data.tofs_e is not None, "no TOF data in this file"

gmd = data.gmd                       # (n_trains, n_bunches) µJ, NaN = no pulse
t_min = (data.tID - data.tID[0]) / TRAIN_RATE_HZ / 60
pulse = np.isfinite(gmd)
if BUNCH_RANGE is None:
    lit_slots = np.flatnonzero(pulse.mean(axis=0) > 0.5)
    BUNCH_RANGE = (int(lit_slots[0]), int(lit_slots[-1]) + 1)
shot = pulse.copy()
shot[:, :BUNCH_RANGE[0]] = False
shot[:, BUNCH_RANGE[1]:] = False

H = (data.tofs_e if DETECTOR == "e" else data.tofs_i)[shot]     # (n_shots, max_hits), 0 = padding
train_idx, slot_idx = np.nonzero(shot)
shots = {
    "gmd": gmd[shot],
    "n_e": (data.tofs_e[shot] > 0).sum(axis=1),
    "n_i": (data.tofs_i[shot] > 0).sum(axis=1),
    "mpe": data.mpe[train_idx],
    "t_min": t_min[train_idx],
    "slot": slot_idx,
}
n_shots = len(H)
print(source, f"| {n_shots} shots, bunch slots {BUNCH_RANGE}, detector {DETECTOR!r}")

In [ ]:
# Distributions of the per-shot values; shaded = filter range
fig, axs = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, name in zip(axs, ["gmd", "n_e", "n_i"]):
    v = shots[name][np.isfinite(shots[name])]
    bins = np.arange(v.max() + 2) - 0.5 if name.startswith("n_") else 100
    ax.hist(v, bins=bins, histtype="step")
    if FILTERS.get(name) is not None:
        ax.axvspan(*FILTERS[name], color="C1", alpha=0.15)
    ax.set(xlabel=name, ylabel="shots", yscale="log" if name.startswith("n_") else "linear")
fig.suptitle(source)
fig.tight_layout()
plt.show()

## 2. Filter

In [ ]:
keep = np.ones(n_shots, bool)
for name, rng in FILTERS.items():
    if rng is not None:
        lo, hi = rng
        keep &= (shots[name] >= lo) & (shots[name] <= hi)
        print(f"{name} in [{lo}, {hi}]: {keep.sum()} shots left")
keep &= np.isfinite(shots[BIN_ON])            # shots without a value can't be binned
print(f"kept {keep.sum()} of {n_shots} shots")

## 3. Average TOF spectra binned on `BIN_ON`

In [ ]:
def edges_for(x, n_bins=None, percentiles=None):
    # BIN_EDGES if given, else n_bins bins spread evenly between two percentiles of x
    if BIN_EDGES is not None and n_bins is None:
        return np.asarray(BIN_EDGES, dtype=float)
    lo, hi = percentiles or BIN_PERCENTILES
    return np.nanpercentile(x, np.linspace(lo, hi, (n_bins or N_BINS) + 1))


def colours(n, cmap="viridis"):
    return mpl.colormaps[cmap](np.linspace(0, 0.95, n))

In [ ]:
tof_edges = np.arange(TOF_RANGE[0], TOF_RANGE[1] + TOF_BIN, TOF_BIN)
tof_cent = 0.5 * (tof_edges[1:] + tof_edges[:-1])
Hk, gk, x = H[keep], shots["gmd"][keep], shots[BIN_ON][keep]
edges = edges_for(x)
cents, in_bin = binning.arb_bool_ar(edges, x)

counts = np.zeros((len(cents), len(tof_cent)))       # summed hits per bin
for k, m in enumerate(in_bin):
    h = Hk[m]
    counts[k] = np.histogram(h[h > 0], tof_edges)[0]
n_in = in_bin.sum(axis=1)
g_in = np.array([np.nansum(gk[m]) for m in in_bin])
all_counts = np.histogram(Hk[Hk > 0], tof_edges)[0]   # all kept shots, for reference

if NORMALISE == "shot":
    spectra, ref, ylabel = counts / n_in[:, None], all_counts / keep.sum(), "hits / shot / bin"
elif NORMALISE == "gmd":
    spectra, ref, ylabel = counts / g_in[:, None], all_counts / np.nansum(gk), "hits / µJ / bin"
elif NORMALISE == "area":
    spectra, ref, ylabel = counts / counts.sum(1, keepdims=True), all_counts / all_counts.sum(), "normalised"
else:
    spectra, ref, ylabel = counts, all_counts, "hits / bin"
print("shots per bin:", n_in.tolist())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for c, sp, col in zip(cents, spectra, colours(len(cents))):
    ax.plot(tof_cent, sp, color=col, lw=0.9)
if NORMALISE is not None:
    ax.plot(tof_cent, ref, color="k", lw=0.8, ls="--", label="all kept shots")
    ax.legend()
sm = mpl.cm.ScalarMappable(mpl.colors.Normalize(cents.min(), cents.max()), "viridis")
fig.colorbar(sm, ax=ax, label=BIN_ON)
ax.set(xlabel="TOF (TDC units)", ylabel=ylabel, yscale="log" if LOG_Y else "linear",
       title=f"{'electrons' if DETECTOR == 'e' else 'ions'} binned on {BIN_ON}, {source}")
fig.tight_layout()
plt.show()

## 4. Change across the bins

Each bin's spectrum divided by the spectrum of all kept shots (same normalisation; per shot when
`NORMALISE = None`), on a coarser TOF axis (`RATIO_REBIN` bins merged) and blank where a cell has fewer
than `RATIO_MIN_COUNTS` hits. With `NORMALISE = "gmd"` and `BIN_ON = "gmd"`, 1 everywhere means a linear
response; features that grow (> 1) or shrink (< 1) with pulse energy stand out.

In [ ]:
R = RATIO_REBIN
nb = (len(tof_cent) // R) * R
c_r = counts[:, :nb].reshape(len(cents), -1, R).sum(axis=-1)     # hits per bin, merged TOF bins
a_r = all_counts[:nb].reshape(-1, R).sum(axis=-1)
edges_r = tof_edges[:nb + 1:R]
if NORMALISE == "gmd":
    num, den = c_r / g_in[:, None], a_r / np.nansum(gk)
elif NORMALISE == "area":
    num, den = c_r / c_r.sum(axis=1, keepdims=True), a_r / a_r.sum()
else:                                                             # None / "shot": compare per shot
    num, den = c_r / n_in[:, None], a_r / keep.sum()
ratio = num / np.where(den > 0, den, np.nan)
ratio[c_r < RATIO_MIN_COUNTS] = np.nan                            # too few hits to judge

fig, ax = plt.subplots(figsize=(14, 4.5))
mesh = ax.pcolormesh(edges_r, edges, ratio, shading="flat", cmap="RdBu_r", vmin=0.5, vmax=1.5)
fig.colorbar(mesh, ax=ax, label="bin / all shots")
ax.set(xlabel="TOF (TDC units)", ylabel=BIN_ON)
fig.tight_layout()
plt.show()